# Imports

In [ ]:

!pip install -U sentence-transformers



In [1]:
import re
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

# Config

In [2]:

QUERIES_PATH = "prepared/queries.csv"
CANDIDATES_PATH = "raw/candidates.csv"

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"


TOP_K = 15
BATCH_SIZE = 64

CAND_EMB_PATH = "embs/embs_raw/candidate_embeddings.npy"
QUERY_EMB_PATH = "embs/embs_raw/query_embeddings.npy"
CAND_INDEX_PATH = "embs/embs_raw/candidate_index.csv"
QUERY_INDEX_PATH = "embs/embs_raw/query_index.csv"
VECTOR_RETRIEVAL_OUT = "embs/retrieval_vector.csv"

USE_GPU = torch.cuda.is_available()


def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = x.replace("\xa0", " ")
    x = re.sub(r"<[^>]+>", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def bool_to_ru(x):
    if pd.isna(x):
        return "не указано"
    if isinstance(x, (bool, np.bool_)):
        return "да" if x else "нет"
    s = str(x).strip().lower()
    if s in {"true", "1", "yes", "да"}:
        return "да"
    if s in {"false", "0", "no", "нет"}:
        return "нет"
    return str(x)

def safe_str(x, default="не указано"):
    x = clean_text(x)
    return x if x else default

def build_candidate_text(row):
    """
    Собираем один текстовый профиль кандидата.
    Не включаем sex / age / has_car по умолчанию.
    """
    parts = [
        f"Желаемая позиция: {safe_str(row.get('desired_position'))}.",
        f"Последняя должность: {safe_str(row.get('last_job_title'))}.",
        f"Последняя компания: {safe_str(row.get('last_company'))}.",
        f"Город: {safe_str(row.get('city'))}.",
        f"Тип занятости: {safe_str(row.get('employment_type'))}.",
        f"График работы: {safe_str(row.get('work_schedule'))}.",
        f"Ожидаемая зарплата: {safe_str(row.get('expected_salary_rub'))} руб.",
        f"Готов к переезду: {bool_to_ru(row.get('ready_to_relocate'))}.",
        f"Готов к командировкам: {bool_to_ru(row.get('ready_for_business_trips'))}.",
        f"Образование: {safe_str(row.get('education_level_and_university'))}.",
        f"Опыт работы: {safe_str(row.get('work_experience'))}.",
    ]
    return " ".join(parts)

def build_query_text(row):
    """
    Для честного baseline берём только query_text.
    Если захочешь экспериментально добавить контекст вакансии,
    можно расширить здесь.
    """
    return safe_str(row.get("query_text"))

def topk_from_similarity(sim_row, k=15):
    """
    sim_row: numpy array shape [num_candidates]
    Возвращает индексы top-k и их score в порядке убывания.
    """
    k = min(k, len(sim_row))
    idx = np.argpartition(-sim_row, kth=k-1)[:k]
    idx = idx[np.argsort(-sim_row[idx])]
    return idx, sim_row[idx]

# Data

In [3]:

queries = pd.read_csv(QUERIES_PATH)
candidates = pd.read_csv(CANDIDATES_PATH)



In [4]:
candidates.head()

,candidate_id,sex,age,expected_salary_rub,desired_position,city,ready_to_relocate,ready_for_business_trips,employment_type,work_schedule,work_experience,work_experience_years,last_company,last_job_title,education_level_and_university,resume_updated_at,has_car
0,C_00001,Мужчина,39,29000,Системный администратор,Советск (Калининградская область),False,False,"частичная занятость, проектная работа, полная ...","гибкий график, полный день, сменный график, ва...",Опыт работы 16 лет 10 месяцев Август 2010 — п...,16.833333,"МАОУ ""СОШ № 1 г.Немана""",Системный администратор,Неоконченное высшее образование 2000 Балтийск...,2019-04-16 15:59:00,True
1,C_00002,Мужчина,60,40000,Технический писатель,Королев,False,True,"частичная занятость, проектная работа, полная ...","гибкий график, полный день, сменный график, уд...",Опыт работы 19 лет 5 месяцев Январь 2000 — по...,19.416667,Временный трудовой коллектив,"Менеджер проекта, Аналитик, Технический писатель",Высшее образование 1981 Военно-космическая ак...,2019-04-12 08:42:00,False
2,C_00003,Женщина,36,20000,Оператор,Тверь,False,False,полная занятость,полный день,Опыт работы 10 лет 3 месяца Октябрь 2004 — Де...,10.250000,ПАО Сбербанк,Кассир-операционист,Среднее специальное образование 2002 Профессио...,2019-04-16 08:35:00,False
3,C_00004,Мужчина,38,100000,Веб-разработчик (HTML / CSS / JS / PHP / базы ...,Саратов,False,True,"частичная занятость, проектная работа, полная ...","гибкий график, удаленная работа",Опыт работы 18 лет 9 месяцев Август 2017 — Ап...,18.750000,OpenSoft,Инженер-программист,Высшее образование 2002 Саратовский государст...,2019-04-08 14:23:00,False
4,C_00005,Женщина,26,140000,Региональный менеджер по продажам,Москва,False,True,полная занятость,полный день,Опыт работы 5 лет 7 месяцев Региональный мене...,5.583333,Мармелад,Менеджер по продажам,Высшее образование 2015 Кгу Психологии и педаг...,2019-04-22 10:32:00,False


In [5]:
# Чистим текстовые колонки
for col in queries.columns:
    queries[col] = queries[col].apply(clean_text)

for col in candidates.columns:
    if candidates[col].dtype == object:
        candidates[col] = candidates[col].apply(clean_text)

In [6]:

candidates["candidate_text"] = candidates.apply(build_candidate_text, axis=1)
queries["query_text_for_embedding"] = queries.apply(build_query_text, axis=1)

print("Candidates:", candidates.shape)
print("Queries:", queries.shape)

print("\nCandidate text example:\n")
print(candidates.loc[0, "candidate_text"][:1200])

print("\nQuery text example:\n")
print(queries.loc[0, "query_text_for_embedding"])



Candidates: (44744, 18)
Queries: (160, 7)

Candidate text example:

Желаемая позиция: Системный администратор. Последняя должность: Системный администратор. Последняя компания: МАОУ "СОШ № 1 г.Немана". Город: Советск (Калининградская область). Тип занятости: частичная занятость, проектная работа, полная занятость. График работы: гибкий график, полный день, сменный график, вахтовый метод, удаленная работа. Ожидаемая зарплата: 29000 руб. Готов к переезду: нет. Готов к командировкам: нет. Образование: Неоконченное высшее образование 2000 Балтийская государственная академия рыбопромыслового флота, Калининград судоводительский, Организация и безопасность движения. Опыт работы: Опыт работы 16 лет 10 месяцев Август 2010 — по настоящее время 8 лет 10 месяцев МАОУ "СОШ № 1 г.Немана" Системный администратор Обслуживание ПК,установка ПО, ремонт, периферийной техники, Интернет локальная сеть. Ведение Электронного журнала, сайта организации. Август 2002 — Август 2010 8 лет 1 месяц ТС "ВЕСТЕР-ИНФО" 

In [8]:
print(candidates['candidate_text'].iloc[0])

Желаемая позиция: Системный администратор. Последняя должность: Системный администратор. Последняя компания: МАОУ "СОШ № 1 г.Немана". Город: Советск (Калининградская область). Тип занятости: частичная занятость, проектная работа, полная занятость. График работы: гибкий график, полный день, сменный график, вахтовый метод, удаленная работа. Ожидаемая зарплата: 29000 руб. Готов к переезду: нет. Готов к командировкам: нет. Образование: Неоконченное высшее образование 2000 Балтийская государственная академия рыбопромыслового флота, Калининград судоводительский, Организация и безопасность движения. Опыт работы: Опыт работы 16 лет 10 месяцев Август 2010 — по настоящее время 8 лет 10 месяцев МАОУ "СОШ № 1 г.Немана" Системный администратор Обслуживание ПК,установка ПО, ремонт, периферийной техники, Интернет локальная сеть. Ведение Электронного журнала, сайта организации. Август 2002 — Август 2010 8 лет 1 месяц ТС "ВЕСТЕР-ИНФО" Старший продавец, директор отдела Продажи компьютерной техники.


# model

In [9]:
device = "cuda" if USE_GPU else "cpu"
model = SentenceTransformer(MODEL_NAME, device=device)

print(f"\nModel: {MODEL_NAME}")
print(f"Device: {device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Device: cuda


# Encode

In [10]:
# ============================================
# ENCODE CANDIDATES
# ============================================
candidate_texts = candidates["candidate_text"].tolist()

candidate_embeddings = model.encode_document(
    candidate_texts,
    batch_size=BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

print("candidate_embeddings shape:", candidate_embeddings.shape)



Batches:   0%|          | 0/700 [00:00<?, ?it/s]

candidate_embeddings shape: (44744, 384)


In [11]:
# ============================================
# ENCODE QUERIES
# ============================================
query_texts = queries["query_text_for_embedding"].tolist()

query_embeddings = model.encode_query(
    query_texts,
    batch_size=BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

print("query_embeddings shape:", query_embeddings.shape)



Batches:   0%|          | 0/3 [00:00<?, ?it/s]

query_embeddings shape: (160, 384)


In [13]:
# ============================================
# SAVE EMBEDDINGS
# ============================================
np.save(CAND_EMB_PATH, candidate_embeddings.astype("float32"))
np.save(QUERY_EMB_PATH, query_embeddings.astype("float32"))

candidates[["candidate_id"]].reset_index(drop=True).to_csv(CAND_INDEX_PATH, index=False, encoding="utf-8-sig")
queries[["query_id"]].reset_index(drop=True).to_csv(QUERY_INDEX_PATH, index=False, encoding="utf-8-sig")

print(f"Saved: {CAND_EMB_PATH}, {QUERY_EMB_PATH}, {CAND_INDEX_PATH}, {QUERY_INDEX_PATH}")



Saved: embs/candidate_embeddings.npy, embs/query_embeddings.npy, embs/candidate_index.csv, embs/query_index.csv


# VECTOR RETRIEVAL

In [14]:
# Так как embeddings уже нормализованы, cosine similarity = dot product
similarity_matrix = np.matmul(query_embeddings, candidate_embeddings.T)
print("similarity_matrix shape:", similarity_matrix.shape)

rows = []

for q_idx, q_row in queries.iterrows():
    t0 = time.perf_counter()

    sim_row = similarity_matrix[q_idx]
    top_idx, top_scores = topk_from_similarity(sim_row, k=TOP_K)

    latency_ms = (time.perf_counter() - t0) * 1000.0
    retrieved_at = datetime.now(timezone.utc).isoformat()

    for rank, (cand_idx, score) in enumerate(zip(top_idx, top_scores), start=1):
        rows.append({
            "query_id": q_row["query_id"],
            "candidate_id": candidates.iloc[cand_idx]["candidate_id"],
            "search_system": "vector",
            "rank": rank,
            "score": float(score),
            "latency_ms": float(latency_ms),
            "retrieved_at": retrieved_at,
        })

retrieval_vector = pd.DataFrame(rows)


similarity_matrix shape: (160, 44744)


In [16]:

assert retrieval_vector["search_system"].eq("vector").all()
assert retrieval_vector["rank"].between(1, TOP_K).all()

# не более 15 строк на query
counts = retrieval_vector.groupby("query_id").size()
assert (counts <= TOP_K).all()

# без дублей candidate_id внутри query
dup_mask = retrieval_vector.duplicated(subset=["query_id", "candidate_id", "search_system"], keep=False)
assert not dup_mask.any(), "Есть дубли candidate_id внутри одного query"

# без дублей rank внутри query
rank_dup_mask = retrieval_vector.duplicated(subset=["query_id", "search_system", "rank"], keep=False)
assert not rank_dup_mask.any(), "Есть дубли rank внутри одного query"

retrieval_vector = retrieval_vector.sort_values(["query_id", "rank"]).reset_index(drop=True)
retrieval_vector.to_csv(VECTOR_RETRIEVAL_OUT, index=False, encoding="utf-8-sig")

print("\nSaved retrieval file:", VECTOR_RETRIEVAL_OUT)
print("Shape:", retrieval_vector.shape)
print(retrieval_vector.head(10))


Saved retrieval file: results_emb/retrieval_vector.csv
Shape: (2400, 7)
  query_id candidate_id search_system  rank     score  latency_ms  \
0  Q_00001      C_21459        vector     1  0.733013    2.050998   
1  Q_00001      C_13965        vector     2  0.709806    2.050998   
2  Q_00001      C_10363        vector     3  0.707725    2.050998   
3  Q_00001      C_37790        vector     4  0.706058    2.050998   
4  Q_00001      C_13979        vector     5  0.703465    2.050998   
5  Q_00001      C_15656        vector     6  0.701935    2.050998   
6  Q_00001      C_44377        vector     7  0.700369    2.050998   
7  Q_00001      C_38834        vector     8  0.699239    2.050998   
8  Q_00001      C_01370        vector     9  0.689442    2.050998   
9  Q_00001      C_28547        vector    10  0.689048    2.050998   

                       retrieved_at  
0  2026-03-29T08:58:32.182143+00:00  
1  2026-03-29T08:58:32.182143+00:00  
2  2026-03-29T08:58:32.182143+00:00  
3  2026-03-29T0

In [17]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
(
    retrieval_vector[retrieval_vector['query_id'] == 'Q_00056']
    .merge(candidates[['candidate_id', 'candidate_text']], how='left', on='candidate_id')
    .merge(queries[['query_id', 'query_text_for_embedding']], how='left', on='query_id')

)[['rank', 'score', 'query_text_for_embedding', 'candidate_text']]

In [19]:
retrieval_vector[retrieval_vector['score'] == retrieval_vector['score'].max()]

,query_id,candidate_id,search_system,rank,score,latency_ms,retrieved_at
825,Q_00056,C_07966,vector,1,0.829182,0.257945,2026-03-29T08:58:32.296097+00:00
